# EDA du jeu d’entraînement — phase 3
Analyse du train uniquement : aucune décision de modèle ne dépend des labels du test.
TARGET=1 signifie défaut. Les variables sensibles sont réservées à l’audit.

In [1]:
import json
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / "params.yaml").exists():
    root = root.parent
train = pd.read_csv(root / "data/interim/train.csv", index_col="SK_ID_CURR")
print(f"Train : {len(train):,} lignes ; taux de défaut : {train.TARGET.mean():.4%}")
print(f"Identifiants uniques : {train.index.is_unique}")

Train : 14,760 lignes ; taux de défaut : 8.0759%
Identifiants uniques : True


In [2]:
missing = train.isna().mean().sort_values(ascending=False).head(12)
print("Principales proportions de valeurs manquantes :")
print(missing.to_string())
print("Sentinelles d’emploi nettoyées :", int(train.EMPLOYED_ANOMALY.sum()))

Principales proportions de valeurs manquantes :
COMMONAREA_AVG              0.700949
COMMONAREA_MODE             0.700949
COMMONAREA_MEDI             0.700949
NONLIVINGAPARTMENTS_AVG     0.696680
NONLIVINGAPARTMENTS_MEDI    0.696680
NONLIVINGAPARTMENTS_MODE    0.696680
FONDKAPREMONT_MODE          0.686992
LIVINGAPARTMENTS_MEDI       0.686992
LIVINGAPARTMENTS_AVG        0.686992
LIVINGAPARTMENTS_MODE       0.686992
FLOORSMIN_MODE              0.679404
FLOORSMIN_AVG               0.679404
Sentinelles d’emploi nettoyées : 2655


In [3]:
for group in ("CODE_GENDER", "AGE_GROUP"):
    print("\nDéfaut observé sur le train par", group)
    print(train.groupby(group, observed=True).TARGET.agg(["count", "mean"]).to_string())


Défaut observé sur le train par CODE_GENDER
             count      mean
CODE_GENDER                 
F             9705  0.068315
M             5055  0.104649

Défaut observé sur le train par AGE_GROUP
           count      mean
AGE_GROUP                 
30-45       5988  0.095357
45-60       4916  0.064280
60+         1701  0.036449
<30         2155  0.112761


In [4]:
manifest = json.loads((root / "data/processed/feature_manifest.json").read_text())
print("Features finales :", manifest["feature_count"])
print("Features numériques :", manifest["numeric_count"])
print("Variables catégorielles encodées :", len(manifest["categorical_inputs"]))
print("Colonnes exclues :", ", ".join(manifest["excluded"]))
print("Règle d’âge :", manifest["age_policy"])

Features finales : 261
Features numériques : 122
Variables catégorielles encodées : 15
Colonnes exclues : AGE_GROUP, AGE_YEARS, CODE_GENDER, DAYS_BIRTH, EMPLOYED_AGE_RATIO, SK_ID_CURR, TARGET
Règle d’âge : EMPLOYED_AGE_RATIO computed for audit only, excluded from model.


Ces différences de défaut observé décrivent les données ; elles ne constituent pas un audit d’équité des décisions du modèle.
Les proxys, les métriques par groupe et la mitigation seront évalués en phase 5.
Le ratio EMPLOYED_AGE_RATIO est calculé dans les exports d’audit et exclu du modèle, conformément au choix utilisateur.
Les catégories et les paramètres d’imputation sont ajustés sur le train uniquement.